In [2]:
# examples/demo_semua_fungsi.py

import mortapy as mp
import math # Diperlukan untuk beberapa verifikasi manual
# from IPython.display import display, Markdown # Tidak dipakai jika ini skrip .py murni

# Fungsi helper untuk mencoba menampilkan objek, default ke print jika .show() tidak ada
# atau jika ingin konsisten dengan print untuk deskripsi
def tampilkan_hasil(deskripsi_awal: str, hasil_objek: mp.ActuarialResult, verifikasi_manual: str = ""):
    print(deskripsi_awal)
    hasil_objek.show() # Ini akan memanggil _repr_latex_ di Jupyter atau print Deskripsi di terminal
    if verifikasi_manual:
        print(verifikasi_manual)
    print("-" * 40)

# --- Definisi Parameter Umum ---
print("=" * 70)
print("              DEMO LENGKAP LIBRARY MORTAPY")
print("=" * 70)

usia_x = 35
usia_y = 60
periode_n_bulat = 5
periode_t_float = 2.75
deferral_m = 3
death_period_u = 10 # <<< VARIABEL INI DITAMBAHKAN
offset_s = 0.5 
suku_bunga = 0.05

print("\n--- Parameter Umum yang Digunakan dalam Demo ---")
print(f"Usia awal (x)         : {usia_x}")
print(f"Suku bunga (i)        : {suku_bunga:.2%}")
print(f"Gender (untuk tabel)  : {gender_pilihan.capitalize() if 'gender_pilihan' in locals() else 'Pria (default)'}") # Penyesuaian jika gender_pilihan belum ada
print(f"Periode bulat (n)       : {periode_n_bulat} tahun")
print(f"Periode non-bulat (t)   : {periode_t_float} tahun")
print(f"Periode tunda (m)       : {deferral_m} tahun")
print(f"Periode kematian (u)    : {death_period_u} tahun") # Sekarang variabel ini ada
print(f"Offset waktu (s)        : {offset_s} tahun")
print("-" * 50)

# Parameter untuk Asumsi (didefinisikan lebih awal untuk kejelasan)
gender_pilihan = 'pria' # Didefinisikan di sini agar tersedia untuk print di atas juga
qx_konstan_val = 0.015
px_konstan_val = 1.0 - qx_konstan_val
omega_dm_val = 100.0
mu_cfm_val = 0.025
gompertz_params_val = [0.00006, 1.085] # B, c
makeham_params_val = [0.00015, 0.00003, 1.095] # A, B, c


# ==============================================================================
# BAGIAN 1: PERHITUNGAN BERBASIS TABEL MORTALITA (TMI DEFAULT)
# ==============================================================================

print("\n" + "=" * 70)
print(" BAGIAN 1: PERHITUNGAN BERBASIS TABEL MORTALITA (TMI DEFAULT)")
print("=" * 70)

try:
    tabel_default = mp.load_default_table()
    print(f"\nBerhasil memuat tabel default: {tabel_default}\n")

    print(f"--- Menggunakan parameter: Usia = {usia_x}, Gender = {gender_pilihan} ---\n")

    # --- 1.1 Probabilitas Hidup (_n p_x) ---
    hasil_tpx_tabel = mp.survival_prob_table(
        age=usia_x, n_years=periode_n_bulat, interest_rate=suku_bunga, gender=gender_pilihan
    )
    tampilkan_hasil(f"1.1 Probabilitas Hidup _{{{periode_n_bulat}}}p_{{{usia_x}}} ({gender_pilihan}, Tabel):", hasil_tpx_tabel)

    # --- 1.2 Probabilitas Kematian (_n q_x) ---
    hasil_tqx_tabel = mp.death_prob_table(
        age=usia_x, n_years=periode_n_bulat, interest_rate=suku_bunga, gender=gender_pilihan
    )
    tampilkan_hasil(f"1.2 Probabilitas Kematian _{{{periode_n_bulat}}}q_{{{usia_x}}} ({gender_pilihan}, Tabel):", hasil_tqx_tabel)

    # --- 1.3 Probabilitas Kematian Ditunda (_m|_u q_x) ---
    hasil_deferred_tabel = mp.deferred_death_prob_table(
        age=usia_x, deferral_period=deferral_m, n_years_death=death_period_u, 
        interest_rate=suku_bunga, gender=gender_pilihan
    )
    tampilkan_hasil(f"1.3 Probabilitas Kematian Ditunda _{{{deferral_m}}}|_{{{death_period_u}}}q_{{{usia_x}}} ({gender_pilihan}, Tabel):", hasil_deferred_tabel)

    # --- 1.4 Force of Mortality (μ_{x+s}) ---
    print("\n1.4 Menghitung Force of Mortality (μ_{x+s}) dari Tabel:") # Diperbaiki agar print hanya sekali
    hasil_fom_cfm_tabel = mp.fom_table(
        age=usia_x, t_offset=offset_s, interest_rate=suku_bunga, 
        gender=gender_pilihan, assumption_fractional='cfm'
    )
    tampilkan_hasil(f"    - Asumsi fraksional CFM, μ_{{{usia_x}+{offset_s}}} ({gender_pilihan}):", hasil_fom_cfm_tabel)
    
    hasil_fom_udd_tabel = mp.fom_table(
        age=usia_x, t_offset=offset_s, interest_rate=suku_bunga, 
        gender=gender_pilihan, assumption_fractional='udd'
    )
    tampilkan_hasil(f"    - Asumsi fraksional UDD, μ_{{{usia_x}+{offset_s}}} ({gender_pilihan}):", hasil_fom_udd_tabel)

    # --- 1.5 PDF Kematian (f_X(x+s) = _s p_x * μ_{x+s}) ---
    hasil_pdf_cfm_tabel = mp.pdf_death_table(
        age=usia_x, t_period=offset_s, interest_rate=suku_bunga, 
        gender=gender_pilihan, assumption_fractional='cfm'
    )
    tampilkan_hasil(f"1.5 PDF Kematian f_X({usia_x}+{offset_s}) ({gender_pilihan}, Tabel, Interpolasi CFM):", hasil_pdf_cfm_tabel)

    # --- 1.6 NSP Whole Life (A_x) ---
    hasil_nsp_wl_tabel = mp.nsp_wl_table(age=usia_x, interest_rate=suku_bunga, gender=gender_pilihan)
    tampilkan_hasil(f"1.6 NSP Whole Life A_{{{usia_x}}} ({gender_pilihan}, Tabel):", hasil_nsp_wl_tabel)

    # --- 1.7 PV Anuitas Whole Life Due (ä_x) ---
    hasil_pv_ann_tabel = mp.pv_annuity_due_wl_table(age=usia_x, interest_rate=suku_bunga, gender=gender_pilihan)
    tampilkan_hasil(f"1.7 PV Anuitas Whole Life Due ä_{{{usia_x}}} ({gender_pilihan}, Tabel):", hasil_pv_ann_tabel)

except FileNotFoundError as e:
    print(f"\n[PERINGATAN] Gagal memuat tabel mortalita default: {e}")
    print("Bagian 1 demo (perhitungan berbasis tabel) akan dilewati.")
    hasil_nsp_wl_tabel = None 
    hasil_pv_ann_tabel = None 
except Exception as e_table:
    print(f"\n[ERROR] Terjadi kesalahan pada perhitungan berbasis tabel: {e_table}")
    hasil_nsp_wl_tabel = None
    hasil_pv_ann_tabel = None


# ==============================================================================
# BAGIAN 2: PERHITUNGAN BERBASIS ASUMSI DISTRIBUSI MURNI
# ==============================================================================
print("\n" + "=" * 70)
print(" BAGIAN 2: PERHITUNGAN BERBASIS ASUMSI DISTRIBUSI MURNI")
print("=" * 70)

assumptions_to_test = [
    ('constant_qx', [qx_konstan_val], f"q_x konstan = {qx_konstan_val}"),
    ('constant_px', [px_konstan_val], f"p_x konstan = {px_konstan_val}"),
    ('de_moivre', [omega_dm_val], f"De Moivre (ω={int(omega_dm_val)})"),
    ('constant_mu_cfm', [mu_cfm_val], f"CFM (μ={mu_cfm_val})"),
    ('gompertz', gompertz_params_val, f"Gompertz (B={gompertz_params_val[0]:.2e}, c={gompertz_params_val[1]})"),
    ('makeham', makeham_params_val, f"Makeham (A={makeham_params_val[0]:.2e}, B={makeham_params_val[1]:.2e}, c={makeham_params_val[2]})")
]

for i, (assumption_type, params, desc_short) in enumerate(assumptions_to_test):
    print(f"\n--- 2.{i+1} Menggunakan Asumsi: {desc_short} ---")
    print(f"   Parameter: Usia = {usia_x}, Periode Non-Bulat = {t_float}, Deferral = {deferral_m}, Death Period = {death_period_u}, Offset = {offset_s}\n")
    
    try:
        # --- Probabilitas Hidup (_t p_x) ---
        hasil_tpx_as = mp.survival_prob_assumption(age=usia_x, period=t_float, interest_rate=suku_bunga, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil(f"   a. Probabilitas Hidup _{{{t_float}}}p_{{{usia_x}}}:", hasil_tpx_as)

        # --- Probabilitas Kematian (_t q_x) ---
        hasil_tqx_as = mp.death_prob_assumption(age=usia_x, period=t_float, interest_rate=suku_bunga, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil(f"   b. Probabilitas Kematian _{{{t_float}}}q_{{{usia_x}}}:", hasil_tqx_as)

        # --- Probabilitas Kematian Ditunda (_m|_u q_x) ---
        hasil_deferred_as = mp.deferred_death_prob_assumption(age=usia_x, deferral_period=float(deferral_m), death_period=float(death_period_u), interest_rate=suku_bunga, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil(f"   c. Probabilitas Kematian Ditunda _{{{deferral_m}}}|_{{{death_period_u}}}q_{{{usia_x}}}:", hasil_deferred_as)

        # --- Force of Mortality (μ_{x+s}) ---
        hasil_fom_as = mp.fom_assumption(age=usia_x, t_offset=offset_s, interest_rate=suku_bunga, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil(f"   d. Force of Mortality μ_{{{usia_x}+{offset_s}}}:", hasil_fom_as)

        # --- PDF Kematian (f_X(x+s)) ---
        hasil_pdf_as = mp.pdf_death_assumption(age=usia_x, t_period=offset_s, interest_rate=suku_bunga, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil(f"   e. PDF Kematian f_X({usia_x}+{offset_s}):", hasil_pdf_as)

        # --- NSP Whole Life (A_x) ---
        hasil_nsp_as = mp.nsp_wl_assumption(age=usia_x, interest_rate=suku_bunga, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil(f"   f. NSP Whole Life A_{{{usia_x}}}:", hasil_nsp_as)
        
        # --- PV Anuitas Whole Life Due (ä_x) ---
        hasil_pv_ann_as = mp.pv_annuity_due_wl_assumption(age=usia_x, interest_rate=suku_bunga, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil(f"   g. PV Anuitas Whole Life Due ä_{{{usia_x}}}:", hasil_pv_ann_as)
        
        if assumption_type == 'de_moivre': # Simpan salah satu hasil untuk Bagian 3
            hasil_nsp_as_dm_untuk_ops = hasil_nsp_as

    except Exception as e_assume:
        print(f"   [ERROR] Terjadi kesalahan pada asumsi {assumption_type}: {e_assume}")


# ==============================================================================
# BAGIAN 3: OPERASI ARITMATIKA PADA HASIL
# ==============================================================================
print("\n" + "=" * 70)
print(" BAGIAN 3: OPERASI ARITMATIKA PADA HASIL `ActuarialResult`")
print("=" * 70)

# Pastikan variabel yang dibutuhkan ada sebelum melakukan operasi
# Inisialisasi dengan None jika perhitungan sebelumnya mungkin gagal
hasil_nsp_as_dm_untuk_ops = locals().get('hasil_nsp_as_dm_untuk_ops', None) 
# hasil_nsp_wl_tabel sudah diinisialisasi None jika gagal

if hasil_nsp_wl_tabel is not None and hasil_nsp_as_dm_untuk_ops is not None:
    print("\nMenjumlahkan NSP dari Tabel (Pria 35) dengan NSP dari Asumsi De Moivre (Usia 35):")
    total_nsp_gabungan = hasil_nsp_wl_tabel + hasil_nsp_as_dm_untuk_ops 
    total_nsp_gabungan.show()
else:
    print("\n[PERINGATAN] Contoh operasi aritmatika tidak dapat dijalankan sepenuhnya.")
    print("Pastikan perhitungan berbasis tabel dan asumsi De Moivre di atas berhasil.")

print("\n" + "=" * 70)
print("              DEMO LENGKAP MORTAPY SELESAI")
print("=" * 70)

              DEMO LENGKAP LIBRARY MORTAPY

--- Parameter Umum yang Digunakan dalam Demo ---
Usia awal (x)         : 35
Suku bunga (i)        : 5.00%
Gender (untuk tabel)  : Pria
Periode bulat (n)       : 5 tahun
Periode non-bulat (t)   : 2.75 tahun
Periode tunda (m)       : 3 tahun
Periode kematian (u)    : 10 tahun
Offset waktu (s)        : 0.5 tahun
--------------------------------------------------

 BAGIAN 1: PERHITUNGAN BERBASIS TABEL MORTALITA (TMI DEFAULT)

Berhasil memuat tabel default: <MortalityTable max_age=111, gender_specific=True, unisex=False>

--- Menggunakan parameter: Usia = 35, Gender = pria ---

1.1 Probabilitas Hidup _{5}p_{35} (pria, Tabel):


<IPython.core.display.Math object>

Deskripsi: Probabilitas Hidup 5 Tahun (Tabel), Usia 35, Gender Pria
----------------------------------------
1.2 Probabilitas Kematian _{5}q_{35} (pria, Tabel):


<IPython.core.display.Math object>

Deskripsi: Probabilitas Kematian 5 Tahun (Tabel), Usia 35, Gender Pria
----------------------------------------
1.3 Probabilitas Kematian Ditunda _{3}|_{10}q_{35} (pria, Tabel):


...

Deskripsi: NSP Whole Life (Asumsi: Makeham (A=0.00015, B=3e-05, c=1.095)), Usia 35
----------------------------------------
   g. PV Anuitas Whole Life Due ä_{35}:


<IPython.core.display.Math object>

Deskripsi: PV Anuitas Whole Life Due (Asumsi: Makeham (A=0.00015, B=3e-05, c=1.095)), Usia 35
----------------------------------------

 BAGIAN 3: OPERASI ARITMATIKA PADA HASIL `ActuarialResult`

Menjumlahkan NSP dari Tabel (Pria 35) dengan NSP dari Asumsi De Moivre (Usia 35):


<IPython.core.display.Math object>

Deskripsi: NSP Asuransi Jiwa Seumur Hidup (Tabel), Usia 35, Gender Pria + NSP Whole Life (Asumsi: De Moivre (ω=100)), Usia 35

              DEMO LENGKAP MORTAPY SELESAI
